In [1]:
# Bibliotecas utilizadas
import os
import pandas as pd
import pyodbc
from datetime import datetime
import warnings

# Pega usuário de rede
usuario = os.getenv('USERNAME')

# Tira mensagens de warning
warnings.filterwarnings('ignore')

# Caminho para o arquivo com credenciais
path = fr'C:\Users\{usuario}\OneDrive - CAIXA Consórcio\Documentos\SENHA_BANCO_DADOS'
os.chdir(path)

# Carrega credenciais
df_senhas = pd.read_excel('SENHAS.xlsx')
server, database, username, password = df_senhas.iloc[0, 0:4]

# Conexão com o SQL Server
conn = pyodbc.connect(
    f'DRIVER={{ODBC Driver 17 for SQL Server}};SERVER={server};DATABASE={database};UID={username};PWD={password}'
)

# Query SQL atualizada com novo período
query_table = """
SELECT DISTINCT
    FT.AnoMesRef,
    FT.ID_Cota,
    FT.ST_Contrato,
    FT.Tipo_Pessoa,
    FT.VL_Bem,
    FT.ST_Contemplacao,
    DP.CD_InscricaoNacional,
    DP.Nome_Pessoa,
    C.DT_Contemplacao,
    C.DT_EntregaBem
FROM 
    FT0015_CarteiraCotas AS FT
LEFT JOIN 
    DM0013_Pessoas AS DP
    ON FT.ID_Pessoa = DP.ID_Pessoa
LEFT JOIN 
    FT0018_Contemplacao AS C
    ON FT.ID_Cota = C.ID_Cota
WHERE 
    FT.AnoMesRef BETWEEN '202201' AND '202504'
    AND FT.ST_Contrato = 'Ativo'
    AND FT.Tipo_Pessoa = 'F'
"""

# Executa a consulta
print("📥 Executando a consulta SQL...")
df = pd.read_sql(query_table, conn)

# Agrupando por cliente e AnoMesRef com contagem distinta de cotas (ID_Cota)
df_agrupado = (
    df.groupby(['CD_InscricaoNacional', 'AnoMesRef'])
      .agg(
          qtd_cotas=('ID_Cota', 'nunique'),     # conta cotas distintas no grupo
          vl_bem_total=('VL_Bem', 'sum'),
          vl_bem_média=('VL_Bem', 'mean'),
          ST_Contrato=('ST_Contrato', lambda x: ', '.join(sorted(set(x)))),
          Tipo_Pessoa=('Tipo_Pessoa', lambda x: ', '.join(sorted(set(x)))),
          Nome_Pessoa=('Nome_Pessoa', lambda x: ', '.join(sorted(set(x)))),
          ST_Contemplacao=('ST_Contemplacao', lambda x: ', '.join(sorted(set(x))))
      )
      .reset_index()
)

# Filtra clientes com média do valor do bem >= 1 milhão
df_filtrado = df_agrupado[df_agrupado['vl_bem_média'] >= 1_000_000]

# Caminho de salvamento
caminho_arquivo = r'C:\Users\GabrielHenriqueSilva\CAIXA Consórcio\Risco de Crédito e Antifraude - Documentos\BUSINESS INTELLIGENCE\DADOS MESA DE DECISÃO\DADOS VALOR DO BEM POR CLIENTE\Dados-Completo-Filtrado.xlsx'

# Exporta para Excel
df_filtrado.to_excel(caminho_arquivo, index=False)

print(f"✅ Arquivo filtrado e salvo com sucesso em:\n{caminho_arquivo}")


📥 Executando a consulta SQL...
✅ Arquivo filtrado e salvo com sucesso em:
C:\Users\GabrielHenriqueSilva\CAIXA Consórcio\Risco de Crédito e Antifraude - Documentos\BUSINESS INTELLIGENCE\DADOS MESA DE DECISÃO\DADOS VALOR DO BEM POR CLIENTE\Dados-Completo-Filtrado.xlsx
